# Train your GPT from scratch on Colab
A self-contained experiment: download Tiny Shakespeare, train a character-level GPT with random weights, compare before/after text, plot losses, and evaluate held-out text.

**Start:** Runtime → Change runtime type → select a GPU → Save. Run cells in order. The first run uses **1,000 optimizer steps**, not 1,000 epochs. After inspecting the results, increase `ADDITIONAL_STEPS` and run the training cell again to continue the same model.

This small model learns character patterns and Shakespeare-like text. It is not instruction-tuned and is not expected to answer questions.

Sources: [dataset repository](https://github.com/karpathy/char-rnn/tree/master/data/tinyshakespeare), [Colab resource limits](https://research.google.com/colaboratory/faq.html). GPU availability and runtime duration vary. No measured training results are embedded in this notebook.

## 1. Check your GPU

In [ ]:
import torch
import matplotlib.pyplot as plt
import time, json, math, shutil
from pathlib import Path
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Select a GPU in Runtime > Change runtime type, then reconnect."
device = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
torch.manual_seed(1337)


## 2. Define the model
Explicit masked attention, learned position embeddings, pre-normalization, residual connections, and next-character cross-entropy. No pretrained weights or tokenizer are loaded.

In [ ]:
import argparse
import math
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F


@dataclass
class GPTConfig:
    vocab_size: int
    context_length: int = 128
    embedding_dim: int = 128
    num_heads: int = 4
    num_layers: int = 4
    dropout: float = 0.1


class CharacterTokenizer:
    def __init__(self, characters):
        self.characters = list(characters)
        self.stoi = {ch: i for i, ch in enumerate(self.characters)}

    def encode(self, text):
        unknown = sorted(set(text) - set(self.stoi))
        if unknown:
            raise ValueError(f"Characters outside the training vocabulary: {unknown!r}")
        return [self.stoi[ch] for ch in text]

    def decode(self, ids):
        return "".join(self.characters[i] for i in ids)


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.num_heads = cfg.num_heads
        self.head_dim = cfg.embedding_dim // cfg.num_heads
        self.qkv = nn.Linear(cfg.embedding_dim, 3 * cfg.embedding_dim)
        self.projection = nn.Linear(cfg.embedding_dim, cfg.embedding_dim)
        self.attention_dropout = nn.Dropout(cfg.dropout)
        self.output_dropout = nn.Dropout(cfg.dropout)
        # True entries are allowed: the diagonal and all earlier positions.
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(cfg.context_length, cfg.context_length, dtype=torch.bool)),
            persistent=False,
        )

    def forward(self, x):
        batch, length, channels = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # (B, T, C) -> (B, H, T, D); C = H * D.
        q, k, v = [
            t.reshape(batch, length, self.num_heads, self.head_dim).transpose(1, 2)
            for t in (q, k, v)
        ]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~self.causal_mask[:length, :length], float("-inf"))
        weights = self.attention_dropout(F.softmax(scores, dim=-1))
        out = weights @ v
        out = out.transpose(1, 2).contiguous().reshape(batch, length, channels)
        return self.output_dropout(self.projection(out))


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.embedding_dim)
        self.attention = CausalSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.embedding_dim, 4 * cfg.embedding_dim),
            nn.GELU(),
            nn.Linear(4 * cfg.embedding_dim, cfg.embedding_dim),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        if min(cfg.vocab_size, cfg.context_length, cfg.embedding_dim,
               cfg.num_heads, cfg.num_layers) < 1:
            raise ValueError("All model dimensions must be positive.")
        if cfg.embedding_dim % cfg.num_heads:
            raise ValueError("embedding_dim must be divisible by num_heads.")
        if not 0 <= cfg.dropout < 1:
            raise ValueError("dropout must be in [0, 1).")
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.embedding_dim)
        self.position_embedding = nn.Embedding(cfg.context_length, cfg.embedding_dim)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg.num_layers)])
        self.final_norm = nn.LayerNorm(cfg.embedding_dim)
        self.output = nn.Linear(cfg.embedding_dim, cfg.vocab_size, bias=False)
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, token_ids, targets=None):
        _, length = token_ids.shape
        if not 1 <= length <= self.cfg.context_length:
            raise ValueError("Sequence length must be between 1 and context_length.")
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        x = self.blocks(self.dropout(x))
        logits = self.output(self.final_norm(x))  # (batch, length, vocabulary)
        loss = None
        if targets is not None:
            # Pass raw logits: cross_entropy applies log-softmax internally.
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, token_ids, new_tokens=300, temperature=0.8, top_k=40):
        if temperature <= 0 or new_tokens < 0 or top_k < 0:
            raise ValueError("temperature must be positive; new_tokens and top_k nonnegative.")
        was_training = self.training
        self.eval()
        try:
            for _ in range(new_tokens):
                context = token_ids[:, -self.cfg.context_length:]
                logits, _ = self(context)
                logits = logits[:, -1, :] / temperature
                if top_k:
                    cutoff = torch.topk(logits, min(top_k, logits.size(-1))).values[:, [-1]]
                    logits = logits.masked_fill(logits < cutoff, float("-inf"))
                next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
                token_ids = torch.cat((token_ids, next_id), dim=1)
            return token_ids
        finally:
            self.train(was_training)


def get_batch(data, context_length, batch_size, device):
    starts = torch.randint(len(data) - context_length, (batch_size,))
    windows = starts[:, None] + torch.arange(context_length)[None, :]
    return data[windows].to(device), data[windows + 1].to(device)


@torch.no_grad()
def evaluate(model, splits, batch_size, eval_batches, device):
    was_training = model.training
    model.eval()
    losses = {}
    try:
        for name, data in splits.items():
            total = 0.0
            for _ in range(eval_batches):
                x, y = get_batch(data, model.cfg.context_length, batch_size, device)
                _, loss = model(x, y)
                total += loss.item()
            losses[name] = total / eval_batches
    finally:
        model.train(was_training)
    return losses


def save_checkpoint(path, model, tokenizer, optimizer, step, losses):
    # Move tensors to CPU so checkpoints are portable across devices.
    payload = {
        "config": asdict(model.cfg),
        "characters": tokenizer.characters,
        "model": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "optimizer": optimizer.state_dict(),
        "step": step,
        "losses": losses,
    }
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)



## 3. Download the dataset and split it
The first 90% is training text, the next 5% validation, and the final 5% test. Validation selects the best checkpoint; test text is reserved for the final evaluation. A character vocabulary is fitted only on training text.

In [ ]:
from urllib.request import urlretrieve
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urlretrieve(DATA_URL, "input.txt")
text = Path("input.txt").read_text(encoding="utf-8")
a, b = int(0.90 * len(text)), int(0.95 * len(text))
texts = {"train": text[:a], "val": text[a:b], "test": text[b:]}
tokenizer = CharacterTokenizer(sorted(set(texts["train"])))
splits = {name: torch.tensor(tokenizer.encode(value), dtype=torch.long)
          for name, value in texts.items()}
print("Characters:", len(text), "| vocabulary:", len(tokenizer.characters))
print({name: len(value) for name, value in splits.items()})
print("Training text preview:\n", texts["train"][:500])


## 4. Configure the experiment
Defaults use approximately 0.8 million parameters. If you encounter GPU out-of-memory errors, reduce `BATCH_SIZE` to 16. Set `SAVE_TO_DRIVE=True` to retain checkpoints in your Google Drive; Colab will request access. Otherwise download the results before the runtime ends.

In [ ]:
BATCH_SIZE = 32
EVAL_INTERVAL = 100
EVAL_BATCHES = 20
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/GPT_from_scratch_run")
else:
    OUT = Path("/content/gpt_results")
OUT.mkdir(parents=True, exist_ok=True)
cfg = GPTConfig(vocab_size=len(tokenizer.characters), context_length=128,
                embedding_dim=128, num_heads=4, num_layers=4, dropout=0.1)
assert min(len(t) for t in splits.values()) > cfg.context_length
model = GPT(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
step = 0
best_val = float("inf")
history = []
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Uniform-random reference loss: {math.log(cfg.vocab_size):.3f} nats/character")
print("Re-running this cell resets the model and optimizer.")


## 5. Record the untrained baseline
Fixed evaluation seeds reuse the same sampled windows at each checkpoint, reducing noise when comparing loss. Evaluation does not change the training random-number state.

In [ ]:
def measure(names=("train", "val"), batches=EVAL_BATCHES):
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(2026)
        return evaluate(model, {n: splits[n] for n in names}, BATCH_SIZE, batches, device)

PROMPT = "ROMEO:\n"
def sample_text(temperature=0.8, count=400):
    ids = torch.tensor([tokenizer.encode(PROMPT)], dtype=torch.long, device=device)
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(42)
        result = model.generate(ids, new_tokens=count, temperature=temperature, top_k=40)
    return tokenizer.decode(result[0].tolist())

baseline = measure()
history.append({"step": 0, **baseline})
before_text = sample_text()
print("Untrained losses:", baseline)
print("Untrained generation:\n", before_text)
(OUT / "before.txt").write_text(before_text, encoding="utf-8")


## 6. Train
Start with 1,000 steps. To continue, change `ADDITIONAL_STEPS` to 4,000 and rerun **only this cell**, then rerun the results cells. The model and optimizer continue from their current state. Each checkpoint also stores optimizer state for later restoration.

In [ ]:
ADDITIONAL_STEPS = 1000
model.train()
start_time = time.perf_counter()
start_step = step
try:
    for _ in range(ADDITIONAL_STEPS):
        x, y = get_batch(splits["train"], cfg.context_length, BATCH_SIZE, device)
        _, loss = model(x, y)
        if not torch.isfinite(loss).item():
            raise RuntimeError("Non-finite loss; lower the learning rate.")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        step += 1
        if step % EVAL_INTERVAL == 0 or step == start_step + ADDITIONAL_STEPS:
            metrics = measure()
            history.append({"step": step, **metrics})
            save_checkpoint(OUT / "last.pt", model, tokenizer, optimizer, step, metrics)
            if metrics["val"] < best_val:
                best_val = metrics["val"]
                save_checkpoint(OUT / "best.pt", model, tokenizer, optimizer, step, metrics)
            (OUT / "history.json").write_text(json.dumps(history, indent=2))
            elapsed = time.perf_counter() - start_time
            remaining = (elapsed / (step - start_step)) * (start_step + ADDITIONAL_STEPS - step)
            print(f"Step {step:5d} | train {metrics['train']:.3f} | val {metrics['val']:.3f} "
                  f"| elapsed {elapsed/60:.1f} min | remaining ~{remaining/60:.1f} min", flush=True)
except KeyboardInterrupt:
    print("Stopped. Resume by rerunning this cell; saved last.pt is the last completed evaluation.")
print("Current step:", step)


## 7. Plot learning curves
Both losses should generally decrease. If training loss keeps falling while validation loss rises, the model is overfitting. The validation estimate uses fixed sampled windows, not the entire split.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([r["step"] for r in history], [r["train"] for r in history], label="Training")
ax.plot([r["step"] for r in history], [r["val"] for r in history], label="Validation")
ax.set(xlabel="Optimizer steps", ylabel="Cross-entropy (nats/character)", title="GPT learning curves")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "loss_curve.png", dpi=150)
plt.show()


## 8. Compare text before and after training
Load the best checkpoint into a separate model so the current training state remains available for continued training. Temperature controls sampling randomness; compare 0.5, 0.8, and 1.0.

In [ ]:
checkpoint = torch.load(OUT / "best.pt", map_location="cpu", weights_only=True)
best_model = GPT(GPTConfig(**checkpoint["config"])).to(device)
best_model.load_state_dict(checkpoint["model"])
best_model.eval()
print("Best checkpoint step:", checkpoint["step"])
print("BEFORE TRAINING:\n", before_text)
samples = {}
for temperature in (0.5, 0.8, 1.0):
    ids = torch.tensor([tokenizer.encode(PROMPT)], dtype=torch.long, device=device)
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(42)
        generated = best_model.generate(ids, 500, temperature, 40)
    samples[str(temperature)] = tokenizer.decode(generated[0].tolist())
    print(f"\nAFTER TRAINING — temperature {temperature}:\n{samples[str(temperature)]}")
(OUT / "samples.json").write_text(json.dumps(samples, indent=2), encoding="utf-8")


## 9. Final held-out test
Run this after choosing your training settings using validation. Loss is averaged over 100 sampled test batches. Perplexity is `exp(loss)` and bits per character is `loss / ln(2)`; lower is better. These character-level values cannot be compared directly with word/subword perplexities. Repeatedly using test results to tune the model makes the test set part of model selection.

In [ ]:
with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
    torch.manual_seed(2027)
    test_loss = evaluate(best_model, {"test": splits["test"]}, BATCH_SIZE, 100, device)["test"]
results = {
    "checkpoint_step": checkpoint["step"],
    "validation_loss": checkpoint["losses"]["val"],
    "test_loss_nats_per_character": test_loss,
    "test_character_perplexity": math.exp(test_loss),
    "test_bits_per_character": test_loss / math.log(2),
    "uniform_character_perplexity": cfg.vocab_size,
    "config": asdict(cfg),
}
print(json.dumps(results, indent=2))
(OUT / "results.json").write_text(json.dumps(results, indent=2))


## 10. Download your results
The ZIP includes best/last checkpoints, loss history, a loss plot, generated text, and test metrics. Download before closing your session if you did not save to Drive.

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/gpt_results_download", "zip", OUT)
files.download(archive)


## Optional: restore training after a disconnected session
If you saved to Drive, run cells 1–4 to recreate the definitions, data, and settings, then run the code below **instead of the baseline cell**, and continue with the training cell. If you downloaded the results ZIP, upload and extract it, then point `OUT` to its directory. This restores weights and optimizer state; it does not reproduce the exact pre-disconnection random-number stream.

```python
checkpoint = torch.load(OUT / "last.pt", map_location=device, weights_only=True)
assert checkpoint["characters"] == tokenizer.characters
cfg = GPTConfig(**checkpoint["config"])
model = GPT(cfg).to(device)
model.load_state_dict(checkpoint["model"])
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
optimizer.load_state_dict(checkpoint["optimizer"])
step = checkpoint["step"]
history = json.loads((OUT / "history.json").read_text())
best_val = min(r["val"] for r in history if r["step"] > 0)
before_text = (OUT / "before.txt").read_text(encoding="utf-8")
PROMPT = "ROMEO:\n"
def measure(names=("train", "val"), batches=EVAL_BATCHES):
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(2026)
        return evaluate(model, {n: splits[n] for n in names}, BATCH_SIZE, batches, device)
```
